# Tech Addiction Prediction: Ultimate Blending Ensemble
In this notebook, we combine the predictions of our finest models: **XGBoost, LightGBM, CatBoost, and a Neural Network (MLP)**. 


In [ ]:
import pandas as pd
import numpy as np
import os


## 1. Define Paths to Notebook Outputs
Ensure these match the exact folder names Kaggle gave to your attached outputs.


In [ ]:
# If your Kaggle notebook URLs were different, you may need to adjust these folder names!
PATH_XGB = '/kaggle/input/pgs-s6e8-xgboost/submission.csv'
PATH_LGB = '/kaggle/input/pgs-s6e8-lightgbm/submission.csv'
PATH_CAT = '/kaggle/input/pgs-s6e8-catboost/submission.csv'
PATH_NN  = '/kaggle/input/pgs-s6e8-neural-network/submission.csv'

# Check if files exist to prevent silent failures
for path, name in zip([PATH_XGB, PATH_LGB, PATH_CAT, PATH_NN], ['XGBoost', 'LightGBM', 'CatBoost', 'Neural Net']):
    if not os.path.exists(path):
        print(f"⚠️ WARNING: {name} submission not found at {path}")
    else:
        print(f"✅ Found {name} submission!")


## 2. Load the Predictions


In [ ]:
xgb_sub = pd.read_csv(PATH_XGB)
lgb_sub = pd.read_csv(PATH_LGB)
cat_sub = pd.read_csv(PATH_CAT)
nn_sub  = pd.read_csv(PATH_NN)

# Verify they all have the same order
assert (xgb_sub['id'] == lgb_sub['id']).all()
assert (xgb_sub['id'] == cat_sub['id']).all()
assert (xgb_sub['id'] == nn_sub['id']).all()

print("All submissions successfully loaded and aligned!")


## 3. The Blend
We assign weights to each model based on their Public LB performance. Better models get higher weight. The Neural Network gets a modest weight to inject diversity.


In [ ]:
WEIGHT_XGB = 0.35
WEIGHT_LGB = 0.30
WEIGHT_CAT = 0.25
WEIGHT_NN  = 0.10

# Ensure weights sum to 1
assert abs(WEIGHT_XGB + WEIGHT_LGB + WEIGHT_CAT + WEIGHT_NN - 1.0) < 1e-6, "Weights must sum to 1!"

final_preds = (
    xgb_sub['addicted_label'] * WEIGHT_XGB +
    lgb_sub['addicted_label'] * WEIGHT_LGB +
    cat_sub['addicted_label'] * WEIGHT_CAT +
    nn_sub['addicted_label']  * WEIGHT_NN
)

submission = pd.DataFrame({
    'id': xgb_sub['id'],
    'addicted_label': final_preds
})

submission.to_csv('submission.csv', index=False)
print("Ensemble submission saved!")
display(submission.head())
